In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "plotting":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / "plotting" / "stage_curves"

import csv
import math
from statistics import NormalDist

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

plt.rcParams["font.family"] = "Charter"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Charter"
plt.rcParams["mathtext.it"] = "Charter:italic"
plt.rcParams["mathtext.bf"] = "Charter:bold"

SERIES_ORDER = ("ippo", "mappo", "pimac_v0", "pimac_v6")
ALGORITHM_LABELS = {
    "ippo": "IPPO",
    "mappo": "MAPPO",
    "pimac_v0": "PIC-MAPPO",
    "pimac_v6": "PC3D",
}
ALGORITHM_COLORS = {
    "ippo": "royalblue",
    "mappo": "#005d5d",
    "pimac_v0": "goldenrod",
    "pimac_v6": "#9f1853",
}
AXIS_LABEL_FONTSIZE = 20
TICKS_LABEL_FONTSIZE = 16
LEGEND_LABEL_FONTSIZE = 18
LINE_Y_PADDING_FRACTION = 0.06

PLOTS_TO_MAKE = [
    "spread_final_selected",
    "lbf_final_selected",
    "rware_final_selected",
]

X_KEY = "global_step"
Y_KEY = "train_return_mean"
SMOOTHING_WINDOW = 250
CI_LEVEL = 0.95
DPI = 300

STAGE_PANEL_COLUMNS = 4
STAGE_PANEL_RELATIVE_X = True
SAVE_LEGEND_SEPARATELY = False
MERGE_IDENTICAL_ADJACENT_STAGES = True
COMPACT_Y_LIMITS = True

SELECTED_CONFIGS = {
    "lbf_final_selected": {
        "title": "LBF",
        "output_filename": "lbf_hard_final_learning_curves.png",
        "task_config_path": "lbf_hard/task.json",
        "results_root": "results/lbf_hard",
        "run_prefix": "final_lbf_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "rware_final_selected": {
        "title": "RWARE",
        "output_filename": "rware_final_learning_curves.png",
        "task_config_path": "robotic_warehouse_dynamic/task.json",
        "results_root": "results/rware",
        "run_prefix": "final_robotic_warehouse_dynamic",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_01",
    },
    "spread_final_selected": {
        "title": "Spread",
        "output_filename": "spread_hard_final_learning_curves.png",
        "task_config_path": "simple_spread_dynamic_hard/task.json",
        "results_root": "results/spread",
        "run_prefix": "final_simple_spread_dynamic_hard",
        "ippo": "best_01",
        "mappo": "best_01",
        "pimac_v0": "best_01",
        "pc3d": "active_03",
    },
}


In [2]:
def build_series(selected):
    results_root = selected["results_root"].rstrip("/")
    run_prefix = selected["run_prefix"]
    configs = {
        "ippo": selected["ippo"],
        "mappo": selected["mappo"],
        "pimac_v0": selected["pimac_v0"],
        "pimac_v6": selected["pc3d"],
    }
    return [
        {
            "label": ALGORITHM_LABELS[algorithm],
            "color": ALGORITHM_COLORS[algorithm],
            "glob": f"{results_root}/{algorithm}/{run_prefix}_{algorithm}_{configs[algorithm]}_s*/train_history.csv",
        }
        for algorithm in SERIES_ORDER
    ]


def stage_output_path(selected):
    base = Path(selected["output_filename"])
    return OUTPUT_DIR / f"{base.stem}_by_stage{base.suffix}"


def legend_output_path(selected):
    base = Path(selected["output_filename"])
    return OUTPUT_DIR / f"{base.stem}_legend{base.suffix}"


def stage_source_path(selected):
    results_root = selected["results_root"].rstrip("/")
    run_prefix = selected["run_prefix"]
    return REPO_ROOT / f"{results_root}/pimac_v6/{run_prefix}_pimac_v6_{selected['pc3d']}_s42/train_history.csv"


In [3]:
def rolling_mean(values, window):
    values = np.asarray(values, dtype=np.float64)
    if window <= 1:
        return values.astype(np.float64, copy=True)
    out = np.empty_like(values, dtype=np.float64)
    cumsum = np.cumsum(values, dtype=np.float64)
    for index in range(values.shape[0]):
        start = max(0, index - window + 1)
        total = cumsum[index] - (cumsum[start - 1] if start > 0 else 0.0)
        out[index] = total / float(index - start + 1)
    return out


def load_history(path):
    rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
    x = np.asarray([float(row[X_KEY]) for row in rows], dtype=np.float64)
    y = np.asarray([float(row[Y_KEY]) for row in rows], dtype=np.float64)
    y = rolling_mean(y, SMOOTHING_WINDOW)
    return x, y


def aggregate_runs(paths, x_min, x_max, relative_to=None):
    runs = [load_history(path) for path in paths]
    reference_x = min(runs, key=lambda run: len(run[0]))[0]
    min_shared_x = max(float(x[0]) for x, _ in runs)
    max_shared_x = min(float(x[-1]) for x, _ in runs)
    min_shared_x = max(min_shared_x, float(x_min))
    max_shared_x = min(max_shared_x, float(x_max))
    reference_x = reference_x[(reference_x >= min_shared_x) & (reference_x <= max_shared_x)]
    if reference_x[0] > min_shared_x:
        reference_x = np.concatenate([np.asarray([min_shared_x], dtype=np.float64), reference_x])
    if reference_x[-1] < max_shared_x:
        reference_x = np.concatenate([reference_x, np.asarray([max_shared_x], dtype=np.float64)])
    reference_x = np.unique(reference_x)
    values = np.vstack([np.interp(reference_x, x, y) for x, y in runs])
    mean = np.mean(values, axis=0)
    stderr = np.std(values, axis=0, ddof=1) / math.sqrt(values.shape[0])
    if relative_to is not None:
        reference_x = reference_x - float(relative_to)
    return reference_x, mean, stderr


def compact_y_limits(curves):
    flat = np.concatenate([np.asarray(curve, dtype=np.float64) for curve in curves])
    y_min = float(np.min(flat))
    y_max = float(np.max(flat))
    span = y_max - y_min
    if span <= 0.0:
        span = max(abs(y_min), 1.0)
    padding = max(1e-6, LINE_Y_PADDING_FRACTION * span)
    return y_min - padding, y_max + padding


In [4]:
def parse_stage_descriptor(raw_name):
    if "__" not in raw_name:
        stage_text = raw_name.replace("stage_", "")
        return ((int(stage_text),) if stage_text.isdigit() else tuple(), tuple())
    stage_prefix, counts_suffix = raw_name.split("__", maxsplit=1)
    stage_text = stage_prefix.replace("stage_", "")
    counts = tuple(int(chunk) for chunk in counts_suffix.split("_") if chunk)
    return ((int(stage_text),), counts)


def format_stage_indices(stage_indices):
    if len(stage_indices) == 1:
        return f"S{stage_indices[0]}"
    return f"S{stage_indices[0]}-{stage_indices[-1]}"


def format_stage_label(stage_indices, counts):
    prefix = format_stage_indices(stage_indices)
    if counts:
        counts_text = ",".join(str(count) for count in counts)
        return rf"{prefix}: $n \in \{{{counts_text}\}}$"
    return prefix


def load_stage_segments(path):
    rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
    segments = []
    current_index = int(rows[0]["stage_index"])
    current_name = rows[0]["stage_name"]
    start_step = float(rows[0][X_KEY])
    previous_step = start_step

    for row in rows[1:]:
        stage_index = int(row["stage_index"])
        step = float(row[X_KEY])
        if stage_index != current_index:
            indices, counts = parse_stage_descriptor(current_name)
            indices = indices or (current_index,)
            segments.append({
                "indices": indices,
                "counts": counts,
                "start": start_step,
                "end": step,
                "label": format_stage_label(indices, counts),
            })
            current_index = stage_index
            current_name = row["stage_name"]
            start_step = step
        previous_step = step

    indices, counts = parse_stage_descriptor(current_name)
    indices = indices or (current_index,)
    segments.append({
        "indices": indices,
        "counts": counts,
        "start": start_step,
        "end": previous_step,
        "label": format_stage_label(indices, counts),
    })
    return segments


def merge_adjacent_stage_segments(segments):
    merged = []
    for segment in segments:
        if (
            merged
            and segment["counts"]
            and merged[-1]["counts"] == segment["counts"]
            and segment["indices"][0] == merged[-1]["indices"][-1] + 1
        ):
            merged[-1]["indices"] = merged[-1]["indices"] + segment["indices"]
            merged[-1]["end"] = segment["end"]
            merged[-1]["label"] = format_stage_label(merged[-1]["indices"], segment["counts"])
        else:
            merged.append(dict(segment))
    return merged


In [5]:
def uses_step_axis():
    return "step" in X_KEY.lower()


def apply_step_axis_format(axis):
    if uses_step_axis():
        axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _pos: f"{value / 1e5:g}"))
        axis.tick_params(axis="x", labelsize=TICKS_LABEL_FONTSIZE)
        axis.tick_params(axis="y", labelsize=TICKS_LABEL_FONTSIZE)


def scaled_step_xlabel(base_label):
    return f"{base_label} (×1e5)"


def display_x_label():
    if X_KEY == "global_step":
        return "Environment Steps"
    return X_KEY.replace("_", " ").title()


def save_legend(selected, series):
    output_path = legend_output_path(selected)
    handles = [Line2D([0], [0], color=item["color"], linewidth=2.0, label=item["label"]) for item in series]
    fig = plt.figure(figsize=(max(2.8, 1.8 * len(handles)), 0.9), dpi=DPI)
    fig.legend(handles=handles, frameon=False, ncol=max(1, len(handles)), loc="center")
    fig.savefig(output_path, bbox_inches="tight", pad_inches=0.05, transparent=True, dpi=DPI)
    plt.close(fig)
    return output_path


In [6]:
def plot_stage_learning_curves(preset_name):
    TOP_FIGURE = preset_name == PLOTS_TO_MAKE[0]
    MIDDLE_FIGURE = preset_name == PLOTS_TO_MAKE[1]
    BOTTOM_FIGURE = preset_name == PLOTS_TO_MAKE[2]
    
    selected = SELECTED_CONFIGS[preset_name]
    series = build_series(selected)
    output_path = stage_output_path(selected)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    segments = load_stage_segments(stage_source_path(selected))
    if MERGE_IDENTICAL_ADJACENT_STAGES:
        segments = merge_adjacent_stage_segments(segments)

    ncols = max(1, min(int(STAGE_PANEL_COLUMNS), len(segments)))
    nrows = int(math.ceil(len(segments) / ncols))
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4 * ncols, 2.5 * nrows),
        dpi=DPI,
        sharey=False,
        squeeze=False,
    )

    for axis in axes.flat[len(segments):]:
        axis.set_visible(False)

    z = NormalDist().inv_cdf(0.5 + CI_LEVEL / 2.0)
    for axis, segment in zip(axes.flat, segments):
        mean_curves = []
        for item in series:
            paths = sorted(REPO_ROOT.glob(item["glob"]))
            x, mean, stderr = aggregate_runs(
                paths,
                x_min=segment["start"],
                x_max=segment["end"],
                relative_to=segment["start"] if STAGE_PANEL_RELATIVE_X else None,
            )
            band = z * stderr
            mean_curves.append(mean)
            axis.plot(x, mean, color=item["color"], linewidth=2.0, label=item["label"])
            axis.fill_between(x, mean - band, mean + band, color=item["color"], alpha=0.18)

        if COMPACT_Y_LIMITS:
            axis.set_ylim(*compact_y_limits(mean_curves))
        axis.set_title(segment["label"], fontsize=15, fontweight="bold")
        axis.grid(True, alpha=0.18)
        apply_step_axis_format(axis)

    if not SAVE_LEGEND_SEPARATELY and TOP_FIGURE:
        handles, labels = axes.flat[0].get_legend_handles_labels()
        fig.legend(handles, labels, frameon=True, ncol=max(1, len(labels)), loc="upper center", bbox_to_anchor=(0.5, 1.15), fontsize=AXIS_LABEL_FONTSIZE)
        fig.tight_layout(rect=(0.01, 0.055, 0.995, 0.895), pad=0.55, w_pad=0.45, h_pad=0.75)
    else:
        fig.tight_layout(rect=(0.01, 0.055, 0.995, 0.925), pad=0.55, w_pad=0.45, h_pad=0.75)

    stage_x_label = "Environment Steps Within Stage" if STAGE_PANEL_RELATIVE_X else display_x_label()
    if MIDDLE_FIGURE:
        fig.supylabel("Mean training return", x=0.0, fontsize=AXIS_LABEL_FONTSIZE)
    else:
        fig.supylabel(" ", x=0.0, fontsize=AXIS_LABEL_FONTSIZE)
    if BOTTOM_FIGURE:
        fig.supxlabel(scaled_step_xlabel(stage_x_label) if uses_step_axis() else stage_x_label, y=-0.1, fontsize=AXIS_LABEL_FONTSIZE)
    fig.savefig(output_path, bbox_inches="tight", dpi=DPI)
    plt.close(fig)

    paths = [output_path]
    if SAVE_LEGEND_SEPARATELY:
        paths.append(save_legend(selected, series))
    return paths


In [7]:
saved_paths = []
for preset_name in PLOTS_TO_MAKE:
    saved_paths.extend(plot_stage_learning_curves(preset_name))

for path in saved_paths:
    print(path)


/Users/akman/pimac/plotting/stage_curves/spread_hard_final_learning_curves_by_stage.png
/Users/akman/pimac/plotting/stage_curves/lbf_hard_final_learning_curves_by_stage.png
/Users/akman/pimac/plotting/stage_curves/rware_final_learning_curves_by_stage.png
